# Week 5 Solutions


## 1) Modified Gram-Schmidt (MGS)

We build a QR factorization of the design matrix $X$ using MGS, then solve the
least squares problem $\min_{\beta} \|X\beta - y\|_2$ via $X = QR$ and
$\beta = R^{-1} Q^T y$.

**Gram-Schmidt orthogonalization (classical):** For columns $x_1, \ldots, x_q$ of $X$,
$$
u_1 = \frac{1}{\|x_1\|_2} x_1.
$$
Given $u_1, \ldots, u_{k-1}$, define
$$
v_k = x_k - \sum_{j=1}^{k-1} (u_j^T x_k)u_j,
\qquad
u_k = \frac{v_k}{\|v_k\|_2}.
$$
The upper-triangular entries of $R$ are
$$
r_{jk} = u_j^T x_k \ \text{for } 1 \le j < k,
\qquad
r_{kk} = \|v_k\|_2,
$$
where $v_k = x_k - \sum_{j=1}^{k-1} r_{jk}u_j$.

**Modified Gram-Schmidt (MGS):**
$$
(X, y) \;=\; (Q, q)
\begin{pmatrix}
R & r \\
0 & d
\end{pmatrix}.
$$

In [1]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(22)

# Parameters
n = 200  # number of observations
p = 20   # number of predictors

# Simulate predictor matrix X from standard normal distribution
X = np.random.randn(n, p)

# Simulate true coefficients beta (can be sparse or dense)
beta_true = np.random.randn(p)

# Simulate noise
epsilon = np.random.randn(n) * 0.5  # standard deviation of noise

# Generate response variable y
y = X @ beta_true + epsilon

print(X.shape)  # Check the shape of X


(200, 20)


In [4]:
def modified_gram_schmidt(A):
    """Return Q, R from the Modified Gram-Schmidt QR factorization."""
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    V = A.copy()

    for i in range(n):
        R[i, i] = np.linalg.norm(V[:, i])
        if R[i, i] == 0:
            raise ValueError('Matrix has linearly dependent columns.')
        Q[:, i] = V[:, i] / R[i, i]
        for j in range(i + 1, n):
            R[i, j] = Q[:, i].T @ V[:, j]
            V[:, j] = V[:, j] - R[i, j] * Q[:, i]

    return Q, R

Q, R = modified_gram_schmidt(X)

# Solve least squares via QR
beta_mgs = np.linalg.solve(R, Q.T @ y)

# Compare with NumPy's least squares
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

print('||beta_mgs - beta_lstsq||_2 =', np.linalg.norm(beta_mgs - beta_lstsq))
print('Residual norm (MGS) =', np.linalg.norm(X @ beta_mgs - y))


||beta_mgs - beta_lstsq||_2 = 7.092513616725673e-15
Residual norm (MGS) = 6.441574010546479


**Example when (classical) Gram-Schmidt can fail**

If columns of $X$ are nearly linearly dependent, roundoff error destroys orthogonality
in the computed $Q$. Classical Gram-Schmidt is especially sensitive; MGS is better but
still suffers when the condition number is extremely large.


In [2]:
# Nearly dependent columns example
np.random.seed(7)
m = 50
v = np.random.randn(m)
A_bad = np.column_stack([v, v + 1e-12*np.random.randn(m), v - 1e-12*np.random.randn(m)])
print(A_bad)


[[ 1.69052570e+00  1.69052570e+00  1.69052570e+00]
 [-4.65937371e-01 -4.65937371e-01 -4.65937371e-01]
 [ 3.28201637e-02  3.28201637e-02  3.28201637e-02]
 [ 4.07516283e-01  4.07516283e-01  4.07516283e-01]
 [-7.88923029e-01 -7.88923029e-01 -7.88923029e-01]
 [ 2.06557291e-03  2.06557291e-03  2.06557291e-03]
 [-8.90385858e-04 -8.90385859e-04 -8.90385859e-04]
 [-1.75472431e+00 -1.75472431e+00 -1.75472431e+00]
 [ 1.01765801e+00  1.01765801e+00  1.01765801e+00]
 [ 6.00498516e-01  6.00498516e-01  6.00498516e-01]
 [-6.25428974e-01 -6.25428974e-01 -6.25428974e-01]
 [-1.71548261e-01 -1.71548261e-01 -1.71548261e-01]
 [ 5.05299374e-01  5.05299374e-01  5.05299374e-01]
 [-2.61356415e-01 -2.61356415e-01 -2.61356415e-01]
 [-2.42749079e-01 -2.42749079e-01 -2.42749079e-01]
 [-1.45324141e+00 -1.45324141e+00 -1.45324141e+00]
 [ 5.54580312e-01  5.54580312e-01  5.54580312e-01]
 [ 1.23880905e-01  1.23880905e-01  1.23880905e-01]
 [ 2.74459924e-01  2.74459924e-01  2.74459924e-01]
 [-1.52652453e+00 -1.52652453e+

In [5]:
def classical_gram_schmidt(A):
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i].T @ v
            v = v - R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] == 0:
            raise ValueError('Matrix has linearly dependent columns.')
        Q[:, j] = v / R[j, j]
    return Q, R

Q_cgs, _ = classical_gram_schmidt(A_bad)
Q_mgs, _ = modified_gram_schmidt(A_bad)

orth_err_cgs = np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(Q_cgs.shape[1]))
orth_err_mgs = np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(Q_mgs.shape[1]))

print('Orthogonality error (CGS) =', orth_err_cgs)
print('Orthogonality error (MGS) =', orth_err_mgs)


Orthogonality error (CGS) = 0.0005981570244860092
Orthogonality error (MGS) = 0.0005981570244860092


In [6]:
np.allclose(Q_cgs, Q_mgs   )

True

## 2) The Rayleigh Quotient

For a (real) symmetric matrix $A$, the Rayleigh quotient is
$r(x) = \frac{x^T A x}{x^T x}$. Its stationary points occur at eigenvectors of $A$,
and the value equals the corresponding eigenvalue.


**Iterative algorithm (power iteration + Rayleigh quotient)**

Given a symmetric matrix $A$ and a nonzero initial vector $x^{(0)}$:

1. Normalize: $x^{(k)} \leftarrow x^{(k)} / \|x^{(k)}\|_2$.
2. Rayleigh quotient: $\lambda^{(k)} = (x^{(k)})^T A x^{(k)}$.
3. Power step: $x^{(k+1)} \leftarrow A x^{(k)}$.
4. Repeat until $|\lambda^{(k)} - \lambda^{(k-1)}|$ is below a tolerance.

In [7]:
# Set random seed
np.random.seed(12)

# Step 1: Generate a random symmetric matrix A (e.g., a covariance-like matrix)
m = 5
A_random = np.random.randn(m, m)
A = A_random.T @ A_random  # ensures A is symmetric positive semi-definite

# Rayleigh-quotient iteration to approximate the dominant eigenvalue
np.random.seed(3)

x = np.random.randn(m)
x = x / np.linalg.norm(x)

max_iter = 1000
tol = 1e-10
lambda_prev = None

for k in range(max_iter):
    # Rayleigh quotient
    lambda_k = (x.T @ A @ x) / (x.T @ x)

    # Convergence check (after first iterate)
    if lambda_prev is not None and abs(lambda_k - lambda_prev) < tol:
        break

    # Power step + normalize
    x = A @ x
    x = x / np.linalg.norm(x)
    lambda_prev = lambda_k

print('Dominant eigenvalue (Rayleigh-quotient iter) =', lambda_k)

Dominant eigenvalue (Rayleigh-quotient iter) = 19.80377835793239


In [8]:

# Eigenvalues/eigenvectors (symmetric => use eigh)
eigvals, eigvecs = np.linalg.eigh(A)

print('Eigenvalues:', eigvals)

# Verify Rayleigh quotient on eigenvectors
rq = []
for i in range(m):
    x = eigvecs[:, i]
    rq_val = (x.T @ A @ x) / (x.T @ x)
    rq.append(rq_val)

print('Rayleigh quotients at eigenvectors:', rq)


Eigenvalues: [1.18324465e-02 1.43184850e+00 2.40179166e+00 3.57717012e+00
 1.98037784e+01]
Rayleigh quotients at eigenvectors: [np.float64(0.01183244654259883), np.float64(1.4318484950513084), np.float64(2.401791659704264), np.float64(3.577170115778478), np.float64(19.80377835793267)]
